In [6]:
import shutil
shutil.rmtree("./delta_warehouse", ignore_errors=True)

In [2]:
import os
import pyspark
from delta import *

delta_warehouse = os.path.abspath("./delta_warehouse")

builder = pyspark.sql.SparkSession.builder.appName("MyApp") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", delta_warehouse) 

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/05/27 20:17:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
schema_nm = "purch_ord"
spark.sql(f"create schema if not exists {schema_nm}")

DataFrame[]

In [4]:
sql = f"""
CREATE OR REPLACE TABLE {schema_nm}.orders (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    total_amount DOUBLE
) USING DELTA
"""
spark.sql(sql)
spark.sql(f"insert into {schema_nm}.orders values (1, current_date(), 1001, 250.75)")

26/05/27 20:17:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[]

In [5]:
spark.sql(f"select * from {schema_nm}.orders").show()

+--------+----------+-----------+------------+
|order_id|order_date|customer_id|total_amount|
+--------+----------+-----------+------------+
|       1|2026-05-27|       1001|      250.75|
+--------+----------+-----------+------------+

